In [3]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

os.makedirs("data", exist_ok=True)
np.random.seed(42)
n_samples = 3500

fault_catalog = [
    ('System_Normal',                'NONE', 0.20),
    ('Spark_Plug_Ignition_Misfire',  'HIGH', 0.09),
    ('Cylinder_2_Injector_Fault',    'HIGH', 0.07),
    ('Fuel_Trim_Lean_Vacuum_Leak',   'MEDIUM_LOW', 0.07),
    ('MAF_Sensor_Circuit_Malfunc',   'MEDIUM_LOW', 0.06),
    ('Cooling_Thermostat_Overheat',  'HIGH', 0.07),
    ('Catalytic_Converter_Wear',     'MEDIUM_LOW', 0.06),
    ('Alternator_Charging_Failure',  'HIGH', 0.08),
    ('Battery_Cell_Degradation',     'MEDIUM_LOW', 0.06),
    ('TPMS_Rapid_Tire_Deflation',    'HIGH', 0.07),
    ('Throttle_Position_Sensor_Err', 'MEDIUM_LOW', 0.05),
    ('Transmission_Solenoid_Stuck',  'HIGH', 0.04),
    ('Turbo_Underboost_Wastegate',   'MEDIUM_LOW', 0.04),
    ('Fuel_Pump_Low_Pressure',       'HIGH', 0.04)
]

states = [x[0] for x in fault_catalog]
probs  = [x[2] for x in fault_catalog]
sev_map = {x[0]: x[1] for x in fault_catalog}
sampled_states = np.random.choice(states, size=n_samples, p=probs)

records = []
for state in sampled_states:
    severity = sev_map[state]
    rpm = np.random.uniform(750, 3600)
    temp = np.random.normal(91.0, 3.8)
    tps = (rpm / 6000.0) * 45.0 + np.random.normal(0, 3.5)
    maf = (rpm * 0.0058) + np.random.normal(0, 1.2)
    stft = np.random.normal(0.0, 3.8)
    batt_v = np.random.normal(12.60, 0.25)
    alt_v  = np.random.normal(14.10, 0.28)
    alt_load = np.random.uniform(25.0, 62.0)
    fl_psi, fr_psi, rl_psi, rr_psi = np.random.normal(32.5, 1.4, 4)
    dtc = 'NONE'

    if state == 'Spark_Plug_Ignition_Misfire':
        rpm = np.random.normal(2450, 480)
        temp = np.random.normal(93.0, 3.5)
        maf = (rpm * 0.0058) + np.random.normal(0, 1.1)
        stft = np.random.normal(18.0, 3.5)
        dtc = np.random.choice(['P0300', 'P0171', 'NONE'], p=[0.82, 0.12, 0.06])
    elif state == 'Cylinder_2_Injector_Fault':
        rpm = np.random.uniform(850, 2900)
        stft = np.random.normal(-14.5, 4.0)
        dtc = np.random.choice(['P0202', 'P0300'], p=[0.78, 0.22])
    elif state == 'Fuel_Trim_Lean_Vacuum_Leak':
        maf = (rpm * 0.0044) + np.random.normal(0, 0.9)
        stft = np.random.normal(15.5, 3.5)
        dtc = np.random.choice(['P0171', 'P0300', 'P0101'], p=[0.76, 0.14, 0.10])
    elif state == 'MAF_Sensor_Circuit_Malfunc':
        maf = np.random.choice([np.random.normal(1.8, 0.8), np.random.normal(44.0, 5.0)])
        stft = np.random.normal(9.0, 6.0)
        dtc = np.random.choice(['P0101', 'P0171'], p=[0.85, 0.15])
    elif state == 'Cooling_Thermostat_Overheat':
        temp = np.random.normal(109.5, 4.5)
        dtc = np.random.choice(['P0217', 'NONE'], p=[0.88, 0.12])
    elif state == 'Catalytic_Converter_Wear':
        rpm = np.random.uniform(1500, 3300)
        temp = np.random.normal(96.0, 3.0)
        stft = np.random.normal(4.0, 3.5)
        dtc = np.random.choice(['P0420', 'NONE'], p=[0.90, 0.10])
    elif state == 'Alternator_Charging_Failure':
        alt_v = np.random.normal(11.8, 0.65)
        batt_v = np.random.normal(11.5, 0.45)
        alt_load = np.random.normal(90.0, 6.0)
        dtc = np.random.choice(['P0562', 'P0560'], p=[0.80, 0.20])
    elif state == 'Battery_Cell_Degradation':
        batt_v = np.random.normal(11.1, 0.50)
        alt_v = np.random.normal(13.8, 0.45)
        alt_load = np.random.normal(74.0, 8.0)
        dtc = np.random.choice(['P0560', 'P0562', 'NONE'], p=[0.75, 0.15, 0.10])
    elif state == 'TPMS_Rapid_Tire_Deflation':
        flat_tire = np.random.choice(['FL', 'FR', 'RL', 'RR'])
        low_val = np.random.normal(20.5, 3.2)
        if flat_tire == 'FL': fl_psi = low_val
        elif flat_tire == 'FR': fr_psi = low_val
        elif flat_tire == 'RL': rl_psi = low_val
        else: rr_psi = low_val
        dtc = np.random.choice(['C0750', 'NONE'], p=[0.92, 0.08])
    elif state == 'Throttle_Position_Sensor_Err':
        tps = np.random.normal(4.5, 2.5)
        rpm = np.random.uniform(1100, 2400)
        dtc = np.random.choice(['P0122', 'P0101'], p=[0.86, 0.14])
    elif state == 'Transmission_Solenoid_Stuck':
        rpm = np.random.normal(3850, 450)
        temp = np.random.normal(99.5, 3.5)
        dtc = np.random.choice(['P0700', 'P0217'], p=[0.88, 0.12])
    elif state == 'Turbo_Underboost_Wastegate':
        rpm = np.random.uniform(2400, 4200)
        maf = (rpm * 0.0040) + np.random.normal(0, 0.8)
        dtc = np.random.choice(['P0299', 'P0101'], p=[0.85, 0.15])
    elif state == 'Fuel_Pump_Low_Pressure':
        rpm = np.random.uniform(1500, 3500)
        stft = np.random.normal(19.5, 3.2)
        maf = (rpm * 0.0055) + np.random.normal(0, 0.9)
        dtc = np.random.choice(['P0087', 'P0300', 'P0171'], p=[0.74, 0.16, 0.10])

    records.append({
        'Engine_RPM': round(max(650, rpm), 0),
        'Coolant_Temp_C': round(temp, 1),
        'MAF_Air_Flow_gs': round(max(0.5, maf), 1),
        'Short_Term_Fuel_Trim_Pct': round(np.clip(stft, -25.0, 25.0), 1),
        'Throttle_Pos_Pct': round(np.clip(tps, 0.0, 100.0), 1),
        'Battery_Voltage_V': round(np.clip(batt_v, 9.0, 15.0), 2),
        'Alternator_Output_V': round(np.clip(alt_v, 9.5, 15.5), 2),
        'Alternator_Load_Pct': round(np.clip(alt_load, 10.0, 100.0), 1),
        'Tire_FL_PSI': round(max(10.0, fl_psi), 1),
        'Tire_FR_PSI': round(max(10.0, fr_psi), 1),
        'Tire_RL_PSI': round(max(10.0, rl_psi), 1),
        'Tire_RR_PSI': round(max(10.0, rr_psi), 1),
        'Min_Tire_PSI': round(max(10.0, min(fl_psi, fr_psi, rl_psi, rr_psi)), 1),
        'DTC_Code': dtc,
        'Root_Cause': state,
        'Severity': severity
    })

df_telemetry = pd.DataFrame(records)
df_telemetry.to_csv("data/obd2_telemetry_dataset.csv", index=False)

workshops_data = pd.DataFrame([
    {"Name": "Mega Dealership (Obour Industrial Zone)", "Slide_Alias": "Mega Dealership", "Type": "Authorized_Dealership", "Supported_Origins": "German,Japanese,Korean,Chinese,European", "City_Zone": "Obour City", "Address": "Block 13004, Industrial Zone A, Obour City", "Phone": "+20 19980 / 02-44891200", "Base_Labor_Cost_EGP": 2200, "Rating_Score": 9.3, "Reviews_Count": 1420, "Top_Comment": "توكيل ممتاز وأجهزة كشف أصلية وتفعيل ضمان فوري، بس الحجز والانتظار بياخد وقت.", "Lat": 30.1982, "Lon": 31.4915},
    {"Name": "AutoWorld Dealer (Ismailia Desert Rd)", "Slide_Alias": "AutoWorld Dealer", "Type": "Authorized_Dealership", "Supported_Origins": "Korean,Chinese,Japanese", "City_Zone": "Ismailia Desert Rd", "Address": "KM 28 Cairo-Ismailia Desert Road, Next to Toll Station", "Phone": "+20 16550 / 01009823411", "Base_Labor_Cost_EGP": 1950, "Rating_Score": 8.9, "Reviews_Count": 980, "Top_Comment": "أفضل توكيل للصيني والكوري على طريق الإسماعيلية وقطع الغيار كلها متوفرة في المخزن.", "Lat": 30.1590, "Lon": 31.4980},
    {"Name": "Bavaria EuroTech Dealership (New Cairo)", "Slide_Alias": "EuroTech Dealer", "Type": "Authorized_Dealership", "Supported_Origins": "German,European", "City_Zone": "New Cairo (5th Settlement)", "Address": "South 90th St, Industrial Triangle, 5th Settlement", "Phone": "+20 19222 / 01223450987", "Base_Labor_Cost_EGP": 2800, "Rating_Score": 9.5, "Reviews_Count": 2150, "Top_Comment": "متخصصين جداً في الألماني وبرمجة الكنترولات والفتيس، السعر غالي بس شغل توكيل مضمون.", "Lat": 30.0155, "Lon": 31.4380},
    {"Name": "City Cars Dealership (Badr Industrial)", "Slide_Alias": "City Cars", "Type": "Authorized_Dealership", "Supported_Origins": "Chinese,Korean,Japanese", "City_Zone": "Badr City", "Address": "3rd Industrial Zone, Badr University Road, Badr City", "Phone": "+20 15880 / 02-28641190", "Base_Labor_Cost_EGP": 1750, "Rating_Score": 7.2, "Reviews_Count": 410, "Top_Comment": "كويس للصيانات الدورية والعربيات اللي في الضمان قريب من بدر والشروق.", "Lat": 30.1365, "Lon": 31.7150},
    {"Name": "West Cairo Authorized Hub (Abu Rawash)", "Slide_Alias": "West Cairo Dealer", "Type": "Authorized_Dealership", "Supported_Origins": "German,Japanese,Korean,Chinese,European", "City_Zone": "6th of October / Abu Rawash", "Address": "KM 28 Cairo-Alex Desert Rd, Abu Rawash Industrial Zone", "Phone": "+20 19011 / 02-35390811", "Base_Labor_Cost_EGP": 2300, "Rating_Score": 9.1, "Reviews_Count": 1890, "Top_Comment": "صرح كبير جداً وفيه ونش إنقاذ سريع، مناسب جداً لو العربية في الضمان.", "Lat": 30.0440, "Lon": 31.0690},
    {"Name": "Pro Workshop (El-Shorouk Auto Zone)", "Slide_Alias": "Pro Workshop", "Type": "Specialized_Center", "Supported_Origins": "Korean,Japanese,Chinese", "City_Zone": "El-Shorouk City", "Address": "Entrance 1 Service Center, El-Shorouk City", "Phone": "01012349876 / 01145678901", "Base_Labor_Cost_EGP": 750, "Rating_Score": 8.5, "Reviews_Count": 640, "Top_Comment": "مهندسين محترمين جداً، كشف بالكمبيوتر دقيق وأسعارهم نص التوكيل للعربيات بره الضمان.", "Lat": 30.1450, "Lon": 31.6020},
    {"Name": "Fix-It Hub (El-Herafieen Gesr El-Suez)", "Slide_Alias": "Fix-It Hub", "Type": "Local_Workshop", "Supported_Origins": "Korean,Japanese,Chinese,European", "City_Zone": "Gesr El-Suez (El-Herafieen)", "Address": "Block 14, Mosque Square, El-Herafieen City, Gesr El-Suez", "Phone": "01287654321 / 01099887766", "Base_Labor_Cost_EGP": 500, "Rating_Score": 7.8, "Reviews_Count": 1120, "Top_Comment": "صنايعية إيدهم تتلف في حرير في المواتير والعفشة للنوبيرا واللانوس والكوري عموماً، وسعرهم حنين.", "Lat": 30.1425, "Lon": 31.3960},
    {"Name": "Deutsch Meister Garage (El-Herafieen)", "Slide_Alias": "Deutsch Garage", "Type": "Specialized_Center", "Supported_Origins": "German,European", "City_Zone": "Gesr El-Suez (El-Herafieen)", "Address": "Street 8, German Auto Sector, El-Herafieen City", "Phone": "01003344556 / 01122334455", "Base_Labor_Cost_EGP": 950, "Rating_Score": 8.8, "Reviews_Count": 830, "Top_Comment": "أحسن بديل للتوكيل لو عربيتك ألماني (BMW / Mercedes / VAG) وخرجت من الضمان.", "Lat": 30.1460, "Lon": 31.3990},
    {"Name": "Ace Repairs (Obour 1st District)", "Slide_Alias": "Ace Repairs", "Type": "Local_Workshop", "Supported_Origins": "Korean,Chinese,Japanese", "City_Zone": "Obour City", "Address": "Service Axis, 1st District, Near Carrefour Obour", "Phone": "01066554433 / 01554433221", "Base_Labor_Cost_EGP": 450, "Rating_Score": 6.8, "Reviews_Count": 310, "Top_Comment": "ورشة قريبة وسريعة لتغيير البوجيهات وتنظيف البوابة والرشاشات.", "Lat": 30.1810, "Lon": 31.4610},
    {"Name": "Dragon China Auto Care (Nasr City)", "Slide_Alias": "Dragon Auto Care", "Type": "Specialized_Center", "Supported_Origins": "Chinese", "City_Zone": "Nasr City", "Address": "El-Hay El-Asher, Auto Maintenance Complex, Nasr City", "Phone": "01022113344 / 01200998877", "Base_Labor_Cost_EGP": 650, "Rating_Score": 8.6, "Reviews_Count": 790, "Top_Comment": "حافظين أعطال الفتيس والتربو للعربيات الصيني (MG, Chery, Geely, BYD) عن ظهر قلب.", "Lat": 30.0510, "Lon": 31.3650},
    {"Name": "Nippon Japan Tech (Maadi)", "Slide_Alias": "Nippon Tech", "Type": "Specialized_Center", "Supported_Origins": "Japanese,Korean", "City_Zone": "Maadi", "Address": "Autostrad Road, Saqr Quraish Industrial Zone, Maadi", "Phone": "01119988223 / 02-27045511", "Base_Labor_Cost_EGP": 700, "Rating_Score": 8.7, "Reviews_Count": 950, "Top_Comment": "ممتازين في الياباني (تويوتا، نيسان، هوندا، ميتسوبيشي) وعندهم قطع غيار استيراد وجديد.", "Lat": 29.9730, "Lon": 31.2840},
    {"Name": "Local Garage (El-Salam Auto Market)", "Slide_Alias": "Local Garage", "Type": "Local_Workshop", "Supported_Origins": "Korean,Chinese,Japanese", "City_Zone": "El-Salam City", "Address": "Ring Road Exit, El-Salam Auto Complex", "Phone": "01234560987", "Base_Labor_Cost_EGP": 400, "Rating_Score": 5.9, "Reviews_Count": 215, "Top_Comment": "سعره رخيص جداً ومناسب للأعطال البسيطة للعربيات القديمة.", "Lat": 30.1610, "Lon": 31.4180},
    {"Name": "Speedy Auto (10th of Ramadan Highway)", "Slide_Alias": "Speedy Auto", "Type": "Local_Workshop", "Supported_Origins": "Korean,Japanese,Chinese", "City_Zone": "10th of Ramadan Rd", "Address": "KM 42 Cairo-Ismailia Road, Near 10th Entrance", "Phone": "01098761234", "Base_Labor_Cost_EGP": 380, "Rating_Score": 4.5, "Reviews_Count": 140, "Top_Comment": "ورشة طوارئ على الطريق السريع لو العربية سخنت أو قطعت سير.", "Lat": 30.2150, "Lon": 31.5640},
    {"Name": "Shubra Electro-Mechanical Center", "Slide_Alias": "Shubra Center", "Type": "Local_Workshop", "Supported_Origins": "Korean,Japanese,European,Chinese", "City_Zone": "Shubra / Downtown", "Address": "Corniche El-Nil, Aghakhan Industrial Strip, Shubra", "Phone": "01277665544 / 02-22056789", "Base_Labor_Cost_EGP": 520, "Rating_Score": 8.1, "Reviews_Count": 680, "Top_Comment": "كهربائي وميكانيكي شاطر جداً في الدينامو والضفيرة والحساسات وبأسعار معقولة.", "Lat": 30.1120, "Lon": 31.2450}
])
workshops_data.to_csv("data/service_centers_registry.csv", index=False)

In [4]:
class PreventiveMaintenanceTracker:
    def __init__(self, current_odometer_km: int, current_date_str: str = "2026-11-18"):
        self.current_odo = current_odometer_km
        self.current_date = datetime.strptime(current_date_str, "%Y-%m-%d")

    def evaluate_service_log(self, service_records: list) -> pd.DataFrame:
        evaluated = []
        for item in service_records:
            km_since_change = self.current_odo - item["Last_Changed_Odo_km"]
            km_remaining = item["Interval_Lifespan_km"] - km_since_change
            wear_pct = round(np.clip((km_since_change / item["Interval_Lifespan_km"]) * 100, 0, 100), 1)
            
            last_date = datetime.strptime(item["Last_Changed_Date"], "%Y-%m-%d")
            days_elapsed = (self.current_date - last_date).days
            days_remaining = item["Interval_Months"] * 30 - days_elapsed
            next_due_date = (last_date + timedelta(days=item["Interval_Months"] * 30)).strftime("%Y-%m-%d")
            next_due_odo = item["Last_Changed_Odo_km"] + item["Interval_Lifespan_km"]

            if km_remaining <= 0 or days_remaining <= 0:
                status = "OVERDUE - CHANGE IMMEDIATELY"
            elif km_remaining <= (item["Interval_Lifespan_km"] * 0.15) or days_remaining <= 20:
                status = "DUE SOON (Upcoming)"
            else:
                status = "HEALTHY (Good Condition)"

            evaluated.append({
                "Component": item["Component"],
                "Category": item["Category"],
                "Last_Changed_Date": item["Last_Changed_Date"],
                "Last_Changed_Odo_km": item["Last_Changed_Odo_km"],
                "Next_Due_Odo_km": next_due_odo,
                "Next_Due_Date": next_due_date,
                "KM_Remaining": km_remaining,
                "Wear_Percentage": f"{wear_pct}%",
                "Est_Replacement_Cost_EGP": item["Est_Cost_EGP"],
                "Status": status,
                "Workshop_Note": item["Notes"]
            })
        
        df_log = pd.DataFrame(evaluated)
        df_log.to_csv("data/vehicle_maintenance_history.csv", index=False)
        return df_log

current_car_odometer = 148500
sample_service_history = [
    {"Component": "Engine Oil & Filter (10,000 KM Synthetic 5W-40)", "Category": "Fluids & Lubrication", "Last_Changed_Odo_km": 139200, "Last_Changed_Date": "2026-05-10", "Interval_Lifespan_km": 10000, "Interval_Months": 6, "Est_Cost_EGP": 2150, "Notes": "Shell Helix Ultra 5W-40 + OEM Oil Filter."},
    {"Component": "Timing Belt & Tensioner Kit (سير الكاتينة والبلية)", "Category": "Engine Mechanical", "Last_Changed_Odo_km": 92000, "Last_Changed_Date": "2024-03-15", "Interval_Lifespan_km": 60000, "Interval_Months": 36, "Est_Cost_EGP": 3400, "Notes": "High risk of belt snap after 60k km."},
    {"Component": "Rear Brake Pads (تيل فرامل خلفي)", "Category": "Braking System", "Last_Changed_Odo_km": 146800, "Last_Changed_Date": "2026-09-20", "Interval_Lifespan_km": 35000, "Interval_Months": 24, "Est_Cost_EGP": 1100, "Notes": "Replaced recently; pads at 95% thickness."},
    {"Component": "Front Brake Pads & Disc Skimming (تيل أمامي)", "Category": "Braking System", "Last_Changed_Odo_km": 118000, "Last_Changed_Date": "2025-04-12", "Interval_Lifespan_km": 30000, "Interval_Months": 20, "Est_Cost_EGP": 1650, "Notes": "Needs immediate replacement."},
    {"Component": "Automatic Transmission Fluid (زيت الفتيس)", "Category": "Transmission", "Last_Changed_Odo_km": 112000, "Last_Changed_Date": "2025-01-18", "Interval_Lifespan_km": 40000, "Interval_Months": 24, "Est_Cost_EGP": 2800, "Notes": "ATF SP-III due within 3,500 km."},
    {"Component": "Spark Plugs Set (طقم بوجيهات)", "Category": "Ignition System", "Last_Changed_Odo_km": 122000, "Last_Changed_Date": "2025-07-05", "Interval_Lifespan_km": 25000, "Interval_Months": 18, "Est_Cost_EGP": 1400, "Notes": "Exceeded 25k km lifespan."},
    {"Component": "12V Lead-Acid Battery (البطارية 70 أمبير)", "Category": "Electrical System", "Last_Changed_Odo_km": 115000, "Last_Changed_Date": "2025-02-01", "Interval_Lifespan_km": 50000, "Interval_Months": 24, "Est_Cost_EGP": 4200, "Notes": "Warranty expires Feb 2027."}
]

tracker = PreventiveMaintenanceTracker(current_odometer_km=current_car_odometer)
df_maintenance = tracker.evaluate_service_log(sample_service_history)
display(df_maintenance[['Component', 'Last_Changed_Date', 'Last_Changed_Odo_km', 'Next_Due_Odo_km', 'KM_Remaining', 'Wear_Percentage', 'Status']])


class DiagnosticGoalAgentV2:
    def __init__(self):
        self.perception_model = RandomForestClassifier(n_estimators=160, max_depth=14, random_state=42)
        self.feature_cols = [
            'Engine_RPM', 'Coolant_Temp_C', 'MAF_Air_Flow_gs',
            'Short_Term_Fuel_Trim_Pct', 'Throttle_Pos_Pct',
            'Battery_Voltage_V', 'Alternator_Output_V', 'Alternator_Load_Pct',
            'Tire_FL_PSI', 'Tire_FR_PSI', 'Tire_RL_PSI', 'Tire_RR_PSI', 'Min_Tire_PSI'
        ]
        self.TARGET_GOAL_STATE = {
            "Mechanical_Integrity": "System_Normal",
            "Safety_Severity": "NONE",
            "Electrical_Charging": "HEALTHY (Charging)",
            "Tire_Stability": "ALL TIRES OK"
        }

    def preprocess(self, df):
        return pd.get_dummies(df[self.feature_cols + ['DTC_Code']])

    def train(self, df):
        X = self.preprocess(df)
        self.encoded_columns = X.columns
        y = df['Root_Cause'] + " | " + df['Severity']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.perception_model.fit(X_train, y_train)
        acc = round(self.perception_model.score(X_test, y_test) * 100, 2)
        print(f"Agent 1 V2 Accuracy: {acc} %")

    def update_internal_state(self, sensor_percept: dict, maintenance_df: pd.DataFrame = None) -> dict:
        df_in = pd.DataFrame([sensor_percept])
        min_psi = min(
            sensor_percept['Tire_FL_PSI'], sensor_percept['Tire_FR_PSI'],
            sensor_percept['Tire_RL_PSI'], sensor_percept['Tire_RR_PSI']
        )
        df_in['Min_Tire_PSI'] = min_psi
        X_in = pd.get_dummies(df_in).reindex(columns=self.encoded_columns, fill_value=0)
        
        pred = self.perception_model.predict(X_in)[0]
        root_cause, severity = pred.split(" | ")

        due_items, bundled_cost = [], 0
        if maintenance_df is not None:
            urgent_df = maintenance_df[maintenance_df['Status'] != "HEALTHY (Good Condition)"]
            due_items = urgent_df['Component'].tolist()
            bundled_cost = int(urgent_df['Est_Replacement_Cost_EGP'].sum())

        return {
            "Root_Cause": root_cause,
            "Severity": severity,
            "Battery_Health": "LOW / DRAINING" if sensor_percept['Battery_Voltage_V'] < 12.0 else "NORMAL",
            "Alternator_Status": "FAULTY CHARGING" if sensor_percept['Alternator_Output_V'] < 13.2 else "HEALTHY (Charging)",
            "TPMS_Alert": f"LOW TIRE PRESSURE ({min_psi} PSI)" if min_psi < 27.0 else "ALL TIRES OK",
            "Overdue_Maintenance_Items": due_items,
            "Bundled_Preventive_Cost_EGP": bundled_cost
        }

    def is_goal_achieved(self, current_state: dict) -> bool:
        return (
            current_state["Root_Cause"] == self.TARGET_GOAL_STATE["Mechanical_Integrity"] and
            current_state["Severity"] == self.TARGET_GOAL_STATE["Safety_Severity"] and
            current_state["Alternator_Status"] == self.TARGET_GOAL_STATE["Electrical_Charging"] and
            current_state["TPMS_Alert"] == self.TARGET_GOAL_STATE["Tire_Stability"]
        )

    def perceive_and_act(self, sensor_percept: dict, maintenance_df: pd.DataFrame = None) -> dict:
        state = self.update_internal_state(sensor_percept, maintenance_df)
        goal_satisfied = self.is_goal_achieved(state)
        state["Goal_Satisfied"] = goal_satisfied

        if goal_satisfied:
            state["Safety_Action"] = "GOAL ACHIEVED: Continue normal trip; only routine preventive schedule applies."
            state["Dispatch_Directive"] = "ROUTINE_MAINTENANCE_ONLY"
        elif state["Severity"] == "HIGH" or "LOW TIRE" in state["TPMS_Alert"] or "FAULTY" in state["Alternator_Status"]:
            state["Safety_Action"] = "STOP VEHICLE & REQUEST TOW / MOBILE TIRE SERVICE"
            state["Dispatch_Directive"] = "EMERGENCY_DISPATCH_REQUIRED"
        else:
            state["Safety_Action"] = "Safe to drive directly to selected workshop to restore normal state."
            state["Dispatch_Directive"] = "STANDARD_REPAIR_DISPATCH"

        return state

agent1_v2 = DiagnosticGoalAgentV2()
agent1_v2.train(df_telemetry)

test_live_sensors = {
    'Engine_RPM': 2450, 'Coolant_Temp_C': 92.0, 'MAF_Air_Flow_gs': 14.2,
    'Short_Term_Fuel_Trim_Pct': 18.5, 'Throttle_Pos_Pct': 21.0,
    'Battery_Voltage_V': 12.6, 'Alternator_Output_V': 14.1, 'Alternator_Load_Pct': 42.0,
    'Tire_FL_PSI': 33.0, 'Tire_FR_PSI': 32.5, 'Tire_RL_PSI': 33.0, 'Tire_RR_PSI': 21.5,
    'DTC_Code': 'P0300'
}

full_diagnosis = agent1_v2.perceive_and_act(test_live_sensors, df_maintenance)
for k, v in full_diagnosis.items():
    print(f"{k}: {v}")

,Component,Last_Changed_Date,Last_Changed_Odo_km,Next_Due_Odo_km,KM_Remaining,Wear_Percentage,Status
0,"Engine Oil & Filter (10,000 KM Synthetic 5W-40)",2026-05-10,139200,149200,700,93.0%,OVERDUE - CHANGE IMMEDIATELY
1,Timing Belt & Tensioner Kit (سير الكاتينة والب...,2024-03-15,92000,152000,3500,94.2%,DUE SOON (Upcoming)
2,Rear Brake Pads (تيل فرامل خلفي),2026-09-20,146800,181800,33300,4.9%,HEALTHY (Good Condition)
3,Front Brake Pads & Disc Skimming (تيل أمامي),2025-04-12,118000,148000,-500,100.0%,OVERDUE - CHANGE IMMEDIATELY
4,Automatic Transmission Fluid (زيت الفتيس),2025-01-18,112000,152000,3500,91.2%,DUE SOON (Upcoming)
5,Spark Plugs Set (طقم بوجيهات),2025-07-05,122000,147000,-1500,100.0%,OVERDUE - CHANGE IMMEDIATELY
6,12V Lead-Acid Battery (البطارية 70 أمبير),2025-02-01,115000,165000,16500,67.0%,HEALTHY (Good Condition)


Agent 1 V2 Accuracy: 95.71 %
Root_Cause: Spark_Plug_Ignition_Misfire
Severity: HIGH
Battery_Health: NORMAL
Alternator_Status: HEALTHY (Charging)
TPMS_Alert: LOW TIRE PRESSURE (21.5 PSI)
Overdue_Maintenance_Items: ['Engine Oil & Filter (10,000 KM Synthetic 5W-40)', 'Timing Belt & Tensioner Kit (سير الكاتينة والبلية)', 'Front Brake Pads & Disc Skimming (تيل أمامي)', 'Automatic Transmission Fluid (زيت الفتيس)', 'Spark Plugs Set (طقم بوجيهات)']
Bundled_Preventive_Cost_EGP: 11400
Goal_Satisfied: False
Safety_Action: STOP VEHICLE & REQUEST TOW / MOBILE TIRE SERVICE
Dispatch_Directive: EMERGENCY_DISPATCH_REQUIRED


In [7]:
class SmartDispatchUtilityAgent:
    ORIGIN_COST_MULTIPLIER = {
        "German": 2.40,
        "European": 1.75,
        "Japanese": 1.40,
        "Korean": 1.05,
        "Chinese": 1.00
    }

    FAULT_BASE_PARTS_EGP = {
        "System_Normal": 0,
        "Spark_Plug_Ignition_Misfire": 1400,
        "Cylinder_2_Injector_Fault": 2800,
        "Fuel_Trim_Lean_Vacuum_Leak": 650,
        "MAF_Sensor_Circuit_Malfunc": 2200,
        "Cooling_Thermostat_Overheat": 1900,
        "Catalytic_Converter_Wear": 6500,
        "Alternator_Charging_Failure": 3600,
        "Battery_Cell_Degradation": 4200,
        "TPMS_Rapid_Tire_Deflation": 450,
        "Throttle_Position_Sensor_Err": 1600,
        "Transmission_Solenoid_Stuck": 5500,
        "Turbo_Underboost_Wastegate": 4800,
        "Fuel_Pump_Low_Pressure": 2900
    }

    def __init__(self, registry_df):
        self.registry = registry_df.copy()

    @staticmethod
    def haversine_km(lat1, lon1, lat2, lon2):
        R = 6371.0
        p1, p2 = np.radians(lat1), np.radians(lat2)
        dp = np.radians(lat2 - lat1)
        dl = np.radians(lon2 - lon1)
        a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
        return np.round(2 * R * np.arcsin(np.sqrt(a)) * 1.25, 1)

    def rank_workshops(self, diagnosis, vehicle_profile, driver_gps):
        df = self.registry.copy()
        origin = vehicle_profile["Origin"]
        year = vehicle_profile["Model_Year"]
        in_warranty = vehicle_profile["In_Warranty"] or (year >= 2023)
        severity = diagnosis["Severity"]
        fault = diagnosis["Root_Cause"]

        df["Distance_km"] = self.haversine_km(driver_gps["Lat"], driver_gps["Lon"], df["Lat"], df["Lon"])

        origin_mult = self.ORIGIN_COST_MULTIPLIER.get(origin, 1.0)
        base_parts = self.FAULT_BASE_PARTS_EGP.get(fault, 1000)

        def compute_cost(row):
            shop_mult = 1.65 if row["Type"] == "Authorized_Dealership" else (1.15 if row["Type"] == "Specialized_Center" else 0.85)
            if in_warranty and row["Type"] == "Authorized_Dealership":
                return 350.0
            total = (base_parts * origin_mult * shop_mult) + (row["Base_Labor_Cost_EGP"] * (1.3 if origin == "German" else 1.0))
            return round(total, -1)

        df["Estimated_Cost_EGP"] = df.apply(compute_cost, axis=1)

        def compute_expertise_match(row):
            supports_origin = origin in row["Supported_Origins"].split(",")
            origin_bonus = 0.15 if supports_origin else -0.35

            if in_warranty:
                base = 0.85 if row["Type"] == "Authorized_Dealership" else 0.15
            elif year <= 2015:
                if row["Type"] == "Local_Workshop":
                    base = 0.82
                elif row["Type"] == "Specialized_Center":
                    base = 0.78
                else:
                    base = 0.25
            else:
                if severity == "HIGH":
                    base = 0.82 if row["Type"] in ["Specialized_Center", "Authorized_Dealership"] else 0.55
                else:
                    base = 0.85 if row["Type"] == "Specialized_Center" else (0.70 if row["Type"] == "Local_Workshop" else 0.40)

            return np.clip(base + origin_bonus, 0.05, 1.0)

        df["Expertise_Match"] = df.apply(compute_expertise_match, axis=1)

        df["Dist_Norm"] = (df["Distance_km"] - df["Distance_km"].min()) / (df["Distance_km"].max() - df["Distance_km"].min() + 1e-5)
        df["Cost_Norm"] = (df["Estimated_Cost_EGP"] - df["Estimated_Cost_EGP"].min()) / (df["Estimated_Cost_EGP"].max() - df["Estimated_Cost_EGP"].min() + 1e-5)
        df["Rating_Norm"] = df["Rating_Score"] / 10.0

        # Increased distance penalty (w2) so nearby workshops (<5 km) beat distant ones (>20 km)
        if in_warranty:
            w1, w2, w3, w4 = 0.48, 0.22, 0.08, 0.22
        elif year <= 2015:
            w1, w2, w3, w4 = 0.34, 0.26, 0.18, 0.22
        else:
            w1, w2, w3, w4 = 0.36, 0.22, 0.18, 0.24

        raw_u = (w1 * df["Expertise_Match"]) - (0.55 * w2 * df["Dist_Norm"]) - (0.55 * w3 * df["Cost_Norm"]) + (w4 * df["Rating_Norm"])
        df["Utility_Score"] = np.round(np.clip(raw_u * 136.8, 30.0, 94.2), 1)

        return df.sort_values(by="Utility_Score", ascending=False)

agent2 = SmartDispatchUtilityAgent(workshops_data)
driver_gps = {"Lat": 30.1650, "Lon": 31.4250}

old_korean_car = {"Brand_Model": "Daewoo Nubira", "Origin": "Korean", "Model_Year": 2006, "In_Warranty": False}
res_old = agent2.rank_workshops(full_diagnosis, old_korean_car, driver_gps)
print("=== CASE A: 2006 Korean Car (Out of Warranty) Top 3 Recommendations ===")
display(res_old[['Name', 'Type', 'City_Zone', 'Distance_km', 'Estimated_Cost_EGP', 'Rating_Score', 'Phone', 'Top_Comment', 'Utility_Score']].head(3))

new_german_car = {"Brand_Model": "BMW 320i", "Origin": "German", "Model_Year": 2025, "In_Warranty": True}
res_new = agent2.rank_workshops(full_diagnosis, new_german_car, driver_gps)
print("\n=== CASE B: 2025 German Car (In Warranty) Top 3 Recommendations ===")
display(res_new[['Name', 'Type', 'City_Zone', 'Distance_km', 'Estimated_Cost_EGP', 'Rating_Score', 'Phone', 'Top_Comment', 'Utility_Score']].head(3))

=== CASE A: 2006 Korean Car (Out of Warranty) Top 3 Recommendations ===


,Name,Type,City_Zone,Distance_km,Estimated_Cost_EGP,Rating_Score,Phone,Top_Comment,Utility_Score
6,Fix-It Hub (El-Herafieen Gesr El-Suez),Local_Workshop,Gesr El-Suez (El-Herafieen),4.7,1750.0,7.8,01287654321 / 01099887766,صنايعية إيدهم تتلف في حرير في المواتير والعفشة...,66.5
8,Ace Repairs (Obour 1st District),Local_Workshop,Obour City,4.9,1700.0,6.8,01066554433 / 01554433221,ورشة قريبة وسريعة لتغيير البوجيهات وتنظيف البو...,63.6
11,Local Garage (El-Salam Auto Market),Local_Workshop,El-Salam City,1.0,1650.0,5.9,01234560987,سعره رخيص جداً ومناسب للأعطال البسيطة للعربيات...,62.8



=== CASE B: 2025 German Car (In Warranty) Top 3 Recommendations ===


,Name,Type,City_Zone,Distance_km,Estimated_Cost_EGP,Rating_Score,Phone,Top_Comment,Utility_Score
0,Mega Dealership (Obour Industrial Zone),Authorized_Dealership,Obour City,9.2,350.0,9.3,+20 19980 / 02-44891200,توكيل ممتاز وأجهزة كشف أصلية وتفعيل ضمان فوري،...,90.6
2,Bavaria EuroTech Dealership (New Cairo),Authorized_Dealership,New Cairo (5th Settlement),20.8,350.0,9.5,+20 19222 / 01223450987,متخصصين جداً في الألماني وبرمجة الكنترولات وال...,87.0
4,West Cairo Authorized Hub (Abu Rawash),Authorized_Dealership,6th of October / Abu Rawash,46.0,350.0,9.1,+20 19011 / 02-35390811,صرح كبير جداً وفيه ونش إنقاذ سريع، مناسب جداً ...,76.5
